In [17]:
import pandas as pd
import numpy as np
import time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [18]:
## CONFIG
USE_PCA = True
PCA_VARIANCE = 0.5

# Parameter budgets for classical-vs-quantum comparison.
# Add/remove values here anytime.
PARAM_BUDGETS = [49, 77]

# If True: only accept models with Actual Parameters <= Target Parameters.
# If False: uses nearest model when under-budget candidate is unavailable.
STRICT_BUDGET = True

RANDOM_STATE = 42

In [19]:
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

X_train = train_df.drop('label', axis=1).values
y_train = train_df['label'].values
X_test = test_df.drop('label', axis=1).values
y_test = test_df['label'].values

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes: {np.unique(y_train)}")

Train: (38, 7129), Test: (34, 7129)
Classes: [0 1]


In [20]:
from sklearn.decomposition import PCA

if USE_PCA:
    pca = PCA(n_components=PCA_VARIANCE)
    X_train_final = pca.fit_transform(X_train)
    X_test_final = pca.transform(X_test)
    print(f"PCA: {X_train.shape[1]} -> {pca.n_components_} features")
    print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.4f}")
else:
    X_train_final = X_train
    X_test_final = X_test
    print(f"No PCA — using all {X_train.shape[1]} features")

PCA: 7129 -> 7 features
Total variance explained: 0.5030


In [21]:
def count_params(model):
    name = type(model).__name__
    if name == 'LogisticRegression':
        return model.coef_.size + model.intercept_.size
    elif name == 'SVC':
        return model.support_vectors_.size + model.intercept_.size + model.dual_coef_.size
    elif name == 'GaussianNB':
        return model.theta_.size + model.var_.size + model.class_prior_.size
    elif name in ('RandomForestClassifier', 'GradientBoostingClassifier'):
        return sum(t.tree_.node_count for e in model.estimators_ for t in (e if hasattr(e, '__iter__') else [e]))
    elif name == 'KNeighborsClassifier':
        return model._fit_X.size
    elif name == 'DecisionTreeClassifier':
        return model.tree_.node_count
    elif name == 'MLPClassifier':
        return sum(w.size for w in model.coefs_) + sum(b.size for b in model.intercepts_)
    elif name == 'XGBClassifier':
        booster = model.get_booster()
        trees = booster.trees_to_dataframe()
        return len(trees)
    elif name == 'LGBMClassifier':
        return model.booster_.num_trees()
    return 'N/A'

In [22]:
def get_model_candidates(random_state=42):
    return {
        'Logistic Regression': [
            LogisticRegression(max_iter=5000, random_state=random_state, C=c)
            for c in [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
        ],
        'SVM (RBF)': [
            SVC(kernel='rbf', random_state=random_state, C=c, gamma=g)
            for c in [0.001, 0.01, 0.1, 1.0, 10.0]
            for g in ['scale', 'auto']
        ],
        'SVM (Linear)': [
            SVC(kernel='linear', random_state=random_state, C=c)
            for c in [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
        ],
        'Gaussian Naive Bayes': [GaussianNB()],
        'Random Forest': [
            RandomForestClassifier(
                n_estimators=n,
                max_depth=d,
                random_state=random_state
            )
            for n in [1, 2, 3, 5, 7, 10, 20]
            for d in [1, 2, 3, 4]
        ],
        'Decision Tree': [
            DecisionTreeClassifier(max_depth=d, random_state=random_state)
            for d in [1, 2, 3, 4, 5, 6, 8, 10, None]
        ],
        'KNN (k=5)': [KNeighborsClassifier(n_neighbors=5)],
        'Gradient Boosting': [
            GradientBoostingClassifier(
                n_estimators=n,
                max_depth=d,
                random_state=random_state
            )
            for n in [1, 2, 3, 5, 7, 10, 20]
            for d in [1, 2, 3]
        ],
        'XGBoost': [
            XGBClassifier(
                n_estimators=n,
                max_depth=d,
                learning_rate=lr,
                eval_metric='logloss',
                random_state=random_state,
                verbosity=0
            )
            for n in [1, 2, 3, 5, 10, 20]
            for d in [1, 2, 3]
            for lr in [0.05, 0.1, 0.3]
        ],
        'LightGBM': [
            LGBMClassifier(
                n_estimators=n,
                max_depth=d,
                learning_rate=lr,
                random_state=random_state,
                verbose=-1
            )
            for n in [1, 2, 3, 5, 10, 20]
            for d in [1, 2, 3, 4, -1]
            for lr in [0.05, 0.1, 0.3]
        ],
        'MLP': [
            MLPClassifier(
                hidden_layer_sizes=hidden,
                max_iter=1000,
                random_state=random_state
            )
            for hidden in [(2,), (3,), (4,), (5,), (6,), (7,), (8,), (10,), (12,), (16,), (4, 2), (6, 3)]
        ],
    }


def fit_best_model_for_budget(model_name, candidates, budget, X, y, strict_budget=True):
    best_under = None
    best_over = None

    for model in candidates:
        try:
            model.fit(X, y)
            params = count_params(model)
            if isinstance(params, str):
                continue

            entry = {
                'model': model,
                'params': int(params),
                'gap': abs(int(params) - budget),
            }

            if params <= budget:
                if (best_under is None) or (entry['gap'] < best_under['gap']):
                    best_under = entry
            else:
                if (best_over is None) or (entry['gap'] < best_over['gap']):
                    best_over = entry
        except Exception as e:
            print(f"[WARN] {model_name} candidate failed: {e}")

    if strict_budget:
        if best_under is None:
            return None, None
        return best_under['model'], best_under['params']

    chosen = best_under if best_under is not None else best_over
    if chosen is None:
        return None, None
    return chosen['model'], chosen['params']

In [23]:
results = []

num_features = X_train_final.shape[1]
pca_variance = round(pca.explained_variance_ratio_.sum(), 4) if USE_PCA else 'N/A'

for budget in PARAM_BUDGETS:
    print(f"\n{'='*22} Target Parameters: {budget} {'='*22}")
    model_candidates = get_model_candidates(random_state=RANDOM_STATE)

    for name, candidates in model_candidates.items():
        start = time.time()
        selected_model, actual_params = fit_best_model_for_budget(
            model_name=name,
            candidates=candidates,
            budget=budget,
            X=X_train_final,
            y=y_train,
            strict_budget=STRICT_BUDGET
        )
        train_time = time.time() - start

        if selected_model is None:
            results.append({
                'Model': name,
                'Target Parameters': budget,
                'Actual Parameters': 'N/A',
                'Within Budget': False,
                'Num Features': num_features,
                'PCA Variance': pca_variance,
                'Train Time (s)': round(train_time, 4),
                'Train Accuracy': 'N/A',
                'Test Accuracy': 'N/A',
            })
            print(f"{name:25s} | Target: {budget:>4} | Actual: {'N/A':>5} | No candidate met budget")
            continue

        train_acc = accuracy_score(y_train, selected_model.predict(X_train_final))
        test_acc = accuracy_score(y_test, selected_model.predict(X_test_final))

        results.append({
            'Model': name,
            'Target Parameters': budget,
            'Actual Parameters': actual_params,
            'Within Budget': actual_params <= budget,
            'Num Features': num_features,
            'PCA Variance': pca_variance,
            'Train Time (s)': round(train_time, 4),
            'Train Accuracy': round(train_acc, 4),
            'Test Accuracy': round(test_acc, 4),
        })

        print(
            f"{name:25s} | Target: {budget:>4} | Actual: {actual_params:>5} "
            f"| Time: {train_time:.4f}s | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}"
        )

results_df = pd.DataFrame(results)
results_df


====================== Target Parameters: 49 ======================
Logistic Regression       | Target:   49 | Actual:     8 | Time: 0.0638s | Train Acc: 0.8158 | Test Acc: 0.5882
SVM (RBF)                 | Target:   49 | Actual:   N/A | No candidate met budget
SVM (Linear)              | Target:   49 | Actual:   N/A | No candidate met budget
Gaussian Naive Bayes      | Target:   49 | Actual:    30 | Time: 0.0011s | Train Acc: 0.9737 | Test Acc: 0.6176
Random Forest             | Target:   49 | Actual:    47 | Time: 0.1329s | Train Acc: 0.9737 | Test Acc: 0.6176
Decision Tree             | Target:   49 | Actual:    11 | Time: 0.0081s | Train Acc: 1.0000 | Test Acc: 0.6471
KNN (k=5)                 | Target:   49 | Actual:   N/A | No candidate met budget
Gradient Boosting         | Target:   49 | Actual:    49 | Time: 0.0599s | Train Acc: 0.9474 | Test Acc: 0.6176
XGBoost                   | Target:   49 | Actual:    35 | Time: 0.2051s | Train Acc: 0.7105 | Test Acc: 0.5882
LightGBM  

d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP                       | Target:   49 | Actual:    46 | Time: 0.9582s | Train Acc: 1.0000 | Test Acc: 0.5294

====================== Target Parameters: 77 ======================
Logistic Regression       | Target:   77 | Actual:     8 | Time: 0.0425s | Train Acc: 0.8158 | Test Acc: 0.5882
SVM (RBF)                 | Target:   77 | Actual:   N/A | No candidate met budget
SVM (Linear)              | Target:   77 | Actual:    73 | Time: 0.0032s | Train Acc: 0.9737 | Test Acc: 0.5588
Gaussian Naive Bayes      | Target:   77 | Actual:    30 | Time: 0.0000s | Train Acc: 0.9737 | Test Acc: 0.6176
Random Forest             | Target:   77 | Actual:    67 | Time: 0.1275s | Train Acc: 0.9737 | Test Acc: 0.6471
Decision Tree             | Target:   77 | Actual:    11 | Time: 0.0031s | Train Acc: 1.0000 | Test Acc: 0.6471
KNN (k=5)                 | Target:   77 | Actual:   N/A | No candidate met budget
Gradient Boosting         | Target:   77 | Actual:    67 | Time: 0.0606s | Train Acc: 0.9474 

d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\__Projects\benchmark-qml-cml\.venv312\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP                       | Target:   77 | Actual:    73 | Time: 0.9652s | Train Acc: 1.0000 | Test Acc: 0.5294


,Model,Target Parameters,Actual Parameters,Within Budget,Num Features,PCA Variance,Train Time (s),Train Accuracy,Test Accuracy
0,Logistic Regression,49,8,True,7,0.503,0.0638,0.8158,0.5882
1,SVM (RBF),49,N/A,False,7,0.503,0.0041,N/A,N/A
2,SVM (Linear),49,N/A,False,7,0.503,0.0069,N/A,N/A
3,Gaussian Naive Bayes,49,30,True,7,0.503,0.0011,0.9737,0.6176
4,Random Forest,49,47,True,7,0.503,0.1329,0.9737,0.6176
5,Decision Tree,49,11,True,7,0.503,0.0081,1.0,0.6471
6,KNN (k=5),49,N/A,False,7,0.503,0.0000,N/A,N/A
7,Gradient Boosting,49,49,True,7,0.503,0.0599,0.9474,0.6176
8,XGBoost,49,35,True,7,0.503,0.2051,0.7105,0.5882
9,LightGBM,49,1,True,7,0.503,0.1461,0.7105,0.5882


In [24]:
output_file = f"classical_param_budget_comparison_PCA={PCA_VARIANCE}.csv"
results_df.to_csv(output_file, index=False)
print(f"Results saved to {output_file}")

Results saved to classical_param_budget_comparison_PCA=0.5.csv
